# Diferentiall express analysis by Sex

In [1]:
import scanpy as sc
import decoupler as dc

# Only needed for processing
import numpy as np
import pandas as pd
from anndata import AnnData
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats


In [2]:
experiment = 'RNAseq_abundances_adjusted_combat_inmose'
sex = 'male'
comparison = 'young.vs.old'

In [15]:
adata = pd.read_csv(f'/home/amore/work/data/{experiment}_gene_symbol_expression.csv', index_col=0)
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,C4orf36.1,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,12741169,0,0,0,0,0,0,2773,0,91.0
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,10663212,0,0,0,0,0,0,1328,0,86.0
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,4248288,0,0,0,0,0,0,1,0,69.0
SRR13758987,14758557,0,12809471,0,0,0,24093106,0,11562944,16086208,...,10497182,0,0,0,0,0,0,1614,0,83.0
SRR13758988,3623061,0,15529838,23957067,0,0,25492722,10773730,12889518,0,...,10506926,0,0,0,0,0,0,795,0,71.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,23486240,0,0,0,0,0,0,0,5351828,27.5
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,34582470,0,0,0,1956002,0,0,0,0,27.5
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,81916604,0,1537700,0,824055,0,1040940,12712840,5952181,27.5


In [11]:
metadata_file = '/home/amore/work/data/All_rna_samples_metadata.csv'
metadata_df = pd.read_csv(metadata_file, index_col=0)
metadata_df

,Age,Status,Experiment,Sex
Run,,,,
SRR13758984,91.0,Sarcopenia,GSE167186,NaN
SRR13758985,86.0,Healthy,GSE167186,male
SRR13758986,69.0,Healthy,GSE167186,male
SRR13758987,83.0,Sarcopenia,GSE167186,NaN
SRR13758988,71.0,UNCLASSIFIED,GSE167186,NaN
...,...,...,...,...
SRR1555214,27.5,untrained,GSE60590,male
SRR1555215,27.5,untrained,GSE60590,male
SRR1555216,27.5,trained,GSE60590,male


In [12]:
metadata_df.drop(columns=['Age','Status','Experiment'], inplace=True)
metadata_df

,Sex
Run,
SRR13758984,NaN
SRR13758985,male
SRR13758986,male
SRR13758987,NaN
SRR13758988,NaN
...,...
SRR1555214,male
SRR1555215,male
SRR1555216,male


In [17]:
adata.head(2)

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,C4orf36.1,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,12741169,0,0,0,0,0,0,2773,0,91.0
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,10663212,0,0,0,0,0,0,1328,0,86.0


In [19]:
adata['Sex'] = metadata_df['Sex']
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,0,0,0,0,0,0,2773,0,91.0,NaN
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,0,0,0,0,0,0,1328,0,86.0,male
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,0,0,0,0,0,0,1,0,69.0,male
SRR13758987,14758557,0,12809471,0,0,0,24093106,0,11562944,16086208,...,0,0,0,0,0,0,1614,0,83.0,NaN
SRR13758988,3623061,0,15529838,23957067,0,0,25492722,10773730,12889518,0,...,0,0,0,0,0,0,795,0,71.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,0,0,0,0,0,0,0,5351828,27.5,male
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,0,0,0,1956002,0,0,0,0,27.5,male
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,0,1537700,0,824055,0,1040940,12712840,5952181,27.5,male


In [20]:
adata = adata[adata['Sex']==sex]
adata = adata.drop(columns='Sex')

In [21]:
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,C4orf36.1,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,10663212,0,0,0,0,0,0,1328,0,86.0
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,4248288,0,0,0,0,0,0,1,0,69.0
SRR13758989,8157761,0,12975244,0,6976439,0,4133727,9553735,8944158,16326912,...,10216566,0,0,0,0,0,0,17171,0,64.0
SRR13758992,9402129,0,10411178,9979835,8660683,0,31061093,10096452,12369261,0,...,10186382,0,0,0,0,0,0,4770,0,80.0
SRR13758998,10752163,0,13844061,0,10528781,0,38447721,15223486,21095132,25317569,...,5275214,0,0,0,0,0,0,23109,0,64.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,23486240,0,0,0,0,0,0,0,5351828,27.5
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,34582470,0,0,0,1956002,0,0,0,0,27.5
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,81916604,0,1537700,0,824055,0,1040940,12712840,5952181,27.5


In [22]:
age_series = adata['Age']
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,C4orf36.1,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,10663212,0,0,0,0,0,0,1328,0,86.0
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,4248288,0,0,0,0,0,0,1,0,69.0
SRR13758989,8157761,0,12975244,0,6976439,0,4133727,9553735,8944158,16326912,...,10216566,0,0,0,0,0,0,17171,0,64.0
SRR13758992,9402129,0,10411178,9979835,8660683,0,31061093,10096452,12369261,0,...,10186382,0,0,0,0,0,0,4770,0,80.0
SRR13758998,10752163,0,13844061,0,10528781,0,38447721,15223486,21095132,25317569,...,5275214,0,0,0,0,0,0,23109,0,64.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,23486240,0,0,0,0,0,0,0,5351828,27.5
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,34582470,0,0,0,1956002,0,0,0,0,27.5
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,81916604,0,1537700,0,824055,0,1040940,12712840,5952181,27.5


In [23]:
adata=adata.astype(int)
adata.isnull().sum().any()

False

In [24]:
ages=adata['Age']
adata=adata.drop('Age',axis=1)
andata=AnnData(adata.to_numpy(), obs=pd.DataFrame(ages))
andata.obs_names = adata.index
andata.var_names = adata.columns


In [25]:
adata = andata #AnnData(adata.to_numpy(), dtype=np.int32)
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 95 × 34327
    obs: 'Age'

In [26]:
# Process treatment information
adata.obs['condition'] = age_series.apply(lambda age: 'young' if age < 35 else ('old' if age >= 65 else 'middle'))


In [27]:
# Obtain genes that pass the thresholds
genes = dc.filter_by_expr(adata, group='condition', min_count=10, min_total_count=15, large_n=1, min_prop=1)

# Filter by these genes
adata = adata[:, genes].copy()
adata

AnnData object with n_obs × n_vars = 95 × 20419
    obs: 'Age', 'condition'

In [28]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=adata,
    design_factors='condition',
    refit_cooks=True,
    inference=inference,
)


In [29]:
dds.deseq2()

Fitting size factors...
... done in 0.04 seconds.

Fitting dispersions...
... done in 14.72 seconds.

Fitting dispersion trend curve...
... done in 0.40 seconds.

Fitting MAP dispersions...
... done in 20.69 seconds.

Fitting LFCs...
... done in 17.91 seconds.

Calculating cook's distance...
... done in 0.16 seconds.

Replacing 2795 outlier genes.

Fitting dispersions...
... done in 1.66 seconds.

Fitting MAP dispersions...
... done in 1.58 seconds.

Fitting LFCs...
... done in 2.82 seconds.



In [30]:
def get_DDS(younger_group, older_group, sex,  save=True):
    comparison = f'{younger_group}.vs.{older_group}'
    stat_res = DeseqStats(
        dds,
        contrast=["condition", younger_group, older_group],
        inference=inference
    )
    stat_res.summary()
    results_df = stat_res.results_df
    if not results_df is None:
        results_df.to_csv(f'/home/amore/work/data/{experiment}_{comparison}_{sex}_DDS.csv', header=True)
    return results_df

In [31]:
get_DDS(younger_group="young", older_group="middle", sex=sex)

Running Wald tests...
... done in 1.56 seconds.



Log2 fold change & Wald test p-value: condition young vs middle
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TSPAN6          5.746886e+07        0.217834  0.146671  1.485193  0.137493   
DPM1            1.009151e+08        0.011014  0.271379  0.040585  0.967626   
SCYL3           1.647659e+07       -1.177177  1.464244 -0.803949  0.421426   
FIRRM           2.637936e+07        1.577602  0.850530  1.854845  0.063618   
CFH             1.646981e+08        0.370897  0.157095  2.360977  0.018227   
...                      ...             ...       ...       ...       ...   
C4orf36.1       4.946470e+07        0.603347  0.458301  1.316487  0.188011   
Unnamed: 34321  5.192041e+05        1.282229  2.532080  0.506393  0.612580   
H2BK1           1.092505e+06       -1.933828  2.714459 -0.712417  0.476206   
Unnamed: 34326  4.458777e+06        3.838778  1.338963  2.866979  0.004144   
TBCEL-TECTA     4.395381e+06       -0.015588  1.301928 -0.011973  0.990447   


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
TSPAN6,5.746886e+07,0.217834,0.146671,1.485193,0.137493,0.496721
DPM1,1.009151e+08,0.011014,0.271379,0.040585,0.967626,0.998639
SCYL3,1.647659e+07,-1.177177,1.464244,-0.803949,0.421426,0.860918
FIRRM,2.637936e+07,1.577602,0.850530,1.854845,0.063618,0.316461
CFH,1.646981e+08,0.370897,0.157095,2.360977,0.018227,0.134699
...,...,...,...,...,...,...
C4orf36.1,4.946470e+07,0.603347,0.458301,1.316487,0.188011,0.595879
Unnamed: 34321,5.192041e+05,1.282229,2.532080,0.506393,0.612580,0.963881
H2BK1,1.092505e+06,-1.933828,2.714459,-0.712417,0.476206,0.900672
Unnamed: 34326,4.458777e+06,3.838778,1.338963,2.866979,0.004144,0.042373


In [32]:
get_DDS(younger_group="young", older_group="old", sex=sex)

Running Wald tests...
... done in 1.59 seconds.



Log2 fold change & Wald test p-value: condition young vs old
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TSPAN6          5.746886e+07        0.015700  0.133204  0.117863  0.906176   
DPM1            1.009151e+08       -0.139787  0.246462 -0.567174  0.570596   
SCYL3           1.647659e+07       -0.716458  1.329804 -0.538769  0.590046   
FIRRM           2.637936e+07       -0.204508  0.772438 -0.264757  0.791197   
CFH             1.646981e+08       -0.039663  0.142671 -0.278006  0.781008   
...                      ...             ...       ...       ...       ...   
C4orf36.1       4.946470e+07       -0.047722  0.416221 -0.114655  0.908719   
Unnamed: 34321  5.192041e+05        1.334708  2.299602  0.580408  0.561639   
H2BK1           1.092505e+06       -1.678799  2.465237 -0.680989  0.495879   
Unnamed: 34326  4.458777e+06        3.877174  1.216025  3.188399  0.001431   
TBCEL-TECTA     4.395381e+06        0.913938  1.182391  0.772957  0.439548   

  

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
TSPAN6,5.746886e+07,0.015700,0.133204,0.117863,0.906176,0.999987
DPM1,1.009151e+08,-0.139787,0.246462,-0.567174,0.570596,0.999987
SCYL3,1.647659e+07,-0.716458,1.329804,-0.538769,0.590046,0.999987
FIRRM,2.637936e+07,-0.204508,0.772438,-0.264757,0.791197,0.999987
CFH,1.646981e+08,-0.039663,0.142671,-0.278006,0.781008,0.999987
...,...,...,...,...,...,...
C4orf36.1,4.946470e+07,-0.047722,0.416221,-0.114655,0.908719,0.999987
Unnamed: 34321,5.192041e+05,1.334708,2.299602,0.580408,0.561639,0.999987
H2BK1,1.092505e+06,-1.678799,2.465237,-0.680989,0.495879,0.999987
Unnamed: 34326,4.458777e+06,3.877174,1.216025,3.188399,0.001431,0.034047


In [33]:
get_DDS(younger_group="middle", older_group="old", sex=sex)

Running Wald tests...
... done in 1.52 seconds.



Log2 fold change & Wald test p-value: condition middle vs old
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TSPAN6          5.746886e+07       -0.202134  0.131672 -1.535141  0.124749   
DPM1            1.009151e+08       -0.150801  0.243627 -0.618983  0.535927   
SCYL3           1.647659e+07        0.460720  1.314505  0.350489  0.725972   
FIRRM           2.637936e+07       -1.782110  0.763552 -2.333974  0.019597   
CFH             1.646981e+08       -0.410560  0.141030 -2.911164  0.003601   
...                      ...             ...       ...       ...       ...   
C4orf36.1       4.946470e+07       -0.651069  0.411433 -1.582441  0.113549   
Unnamed: 34321  5.192041e+05        0.052479  2.273140  0.023087  0.981581   
H2BK1           1.092505e+06        0.255029  2.436869  0.104655  0.916650   
Unnamed: 34326  4.458777e+06        0.038396  1.202035  0.031943  0.974518   
TBCEL-TECTA     4.395381e+06        0.929526  1.168788  0.795291  0.426444   

 

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
TSPAN6,5.746886e+07,-0.202134,0.131672,-1.535141,0.124749,0.407441
DPM1,1.009151e+08,-0.150801,0.243627,-0.618983,0.535927,0.884745
SCYL3,1.647659e+07,0.460720,1.314505,0.350489,0.725972,0.966403
FIRRM,2.637936e+07,-1.782110,0.763552,-2.333974,0.019597,0.123200
CFH,1.646981e+08,-0.410560,0.141030,-2.911164,0.003601,0.035213
...,...,...,...,...,...,...
C4orf36.1,4.946470e+07,-0.651069,0.411433,-1.582441,0.113549,0.386361
Unnamed: 34321,5.192041e+05,0.052479,2.273140,0.023087,0.981581,0.998841
H2BK1,1.092505e+06,0.255029,2.436869,0.104655,0.916650,0.994458
Unnamed: 34326,4.458777e+06,0.038396,1.202035,0.031943,0.974518,0.997728
